# Lesson 04 — 给计时器一张状态图

这是一段真实的 C++ 实验：你修改代码格，再运行，就会看到下方深色虚拟设备的输入、输出和检查结果变化。Python 只负责让 VS Code Notebook 启动 C++ 编译和实验台。

## 先运行

按顺序运行初始化格和后面的代码格。绿色 ✓ 表示本格 C++ 符合当前契约；红色 ✗ 会给出预期与实际输出。


In [ ]:
# 运行支架：只运行一次，不需要学习或修改 Python。
from pathlib import Path
project = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "tools/notebook_setup.py").is_file()), None)
if project is None:
    raise RuntimeError("请用 VS Code 打开 esp32-hands-on-academy 文件夹")
get_ipython().run_line_magic("run", str(project / "tools/notebook_setup.py"))


当工具有 start、pause、resume、reset 和自动完成时，用几个互相覆盖的 bool 很快会失控。我们把设备的合法状态和转换写清楚。

## 最小知识

有限状态机（state machine）让每个时刻只有一个状态。命令从 `input.text` 进入：`start`、`pause`、`resume`、`reset`。同一条命令不会因多次 tick 重复执行。


## 1. 第一个实验（Guided）

先运行。不要急着读每一行；先操作实验台，观察输入与输出之间的关系。


In [ ]:
%%academy_lab 04-state-machine guided guided pass
#pragma once
#include "academy/device.hpp"
#include <string>
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& b) : b_(b) {
        }
        void tick() {
            if (b_.input.text == "start" && last_ != "start") {
                b_.output.state = "running";
                b_.output.led = true;
            }
            last_ = b_.input.text;
        }

    private:
        Device& b_;
        std::string last_;
    };
}


## 2. 修改一次

把完成时间由 `1000` 改为 `2000`。想一想 pause 后再 resume 时，原来的 deadline 为什么不能直接继续使用？

重新运行这一个格，再在实验台确认你的预测。


In [ ]:
%%academy_lab 04-state-machine modify
#pragma once
#include "academy/device.hpp"
#include <string>
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& b) : b_(b) {
        }
        void tick() {
            if (b_.input.text == "start" && last_ != "start") {
                on_ = true;
                end_ = b_.input.now_ms + 2000;
                b_.output.state = "running";
            }
            if (on_ && b_.input.now_ms >= end_) {
                on_ = false;
                b_.output.state = "done";
            }
            b_.output.led = on_;
            last_ = b_.input.text;
        }

    private:
        Device& b_;
        std::string last_;
        bool on_ = false;
        unsigned end_ = 0;
    };
}


## 3. 故障实验（Debug）

下面代码只要收到 start 就 running，pause、reset 和完成条件相互覆盖。运行后失败是预期现象。用面板的时间线、波形、事务或终端找出失败发生在何时。


In [ ]:
%%academy_lab 04-state-machine broken exercise fail
#pragma once
#include "academy/device.hpp"
#include <string>
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& b) : b_(b) {
        }
        void tick() {
            if (b_.input.text == "start")
                state_ = "running";
            if (b_.input.text == "pause")
                state_ = "paused";
            if (b_.input.now_ms >= end_)
                state_ = "done";
            b_.output.state = state_;
        }

    private:
        Device& b_;
        std::string state_ = "idle";
        unsigned end_ = 0;
    };
}


## 4. 你来完成（Build）

实现 idle → running → paused → running / done 的转换。暂停要保存剩余时间；reset 从任何状态回到 idle。

默认这格会失败，失败信息是需求的一部分。先自己尝试；需要时再看 [分层 Hint](../lessons/04-state-machine/hints.md)。


In [ ]:
%%academy_lab 04-state-machine exercise exercise fail
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& b) : b_(b) {
        }
        void tick() { // TODO: 处理 start / pause / resume / reset
            b_.output.state = "idle";
        }

    private:
        Device& b_;
    };
}


## 5. 开放挑战

增加一个 `alarm` 状态，要求只有用户确认后才回 idle。为每一条非法命令决定“忽略”还是“报错”。

## 你刚刚真正学到了什么？

状态机减少了组合爆炸：状态名表达意图，转换集中处理，输出从当前状态派生。

## 硬件迁移

STEP 2 可以让按钮产生命令、让显示器显示状态。状态机本身保持不变；GPIO、屏幕或网络只是输入输出适配器。

完成自己的尝试后才查看 [独立 Solution Notebook](solutions/04-state-machine-solution.ipynb)。
